<a href="https://colab.research.google.com/github/Saksham1SK/CIFAR10-Explorations/blob/main/CIFAR_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [17]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

print("Chronological Step 2 Complete! Conveyor belts are fully loaded.")

Chronological Step 2 Complete! Conveyor belts are fully loaded.


In [10]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [11]:
net = BaselineCNN().to(device)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr = 0.005, momentum = 0.9)

In [13]:
for epoch in range(5):
  running_loss = 0.0
  for i, data in enumerate(trainloader, 0):
    inputs, labels = data[0].to(device), data[1].to(device)
    optimizer.zero_grad()
    outputs = net(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()
  print(f"Epoch {epoch + 1} finished. Average Loss: {running_loss / len(trainloader):.4f}")
print("Baseline Classification Training Complete!")

Epoch 1 finished. Average Loss: 1.5829
Epoch 2 finished. Average Loss: 1.1360
Epoch 3 finished. Average Loss: 0.9528
Epoch 4 finished. Average Loss: 0.8307
Epoch 5 finished. Average Loss: 0.7301
Baseline Classification Training Complete!


In [20]:
def create_jigsaw_puzzle(image_tensor):
    img = image_tensor.clone()

    top_left  = img[:, 0:16, 0:16]
    top_right = img[:, 0:16, 16:32]
    bot_left  = img[:, 16:32, 0:16]
    bot_right = img[:, 16:32, 16:32]

    tiles = [top_left, top_right, bot_left, bot_right]
    random.shuffle(tiles)

    scrambled_top = torch.cat((tiles[0], tiles[1]), dim=2)
    scrambled_bot = torch.cat((tiles[2], tiles[3]), dim=2)

    scrambled_image = torch.cat((scrambled_top, scrambled_bot), dim=1)

    return scrambled_image

correct_clean = 0
correct_scrambled = 0
total = 0

net.eval()
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)

        outputs_clean = net(images)
        _, pred_clean = torch.max(outputs_clean.data, 1)
        correct_clean += (pred_clean == labels).sum().item()

        scrambled_batch = torch.stack([create_jigsaw_puzzle(img) for img in data[0]]).to(device)
        outputs_scrambled = net(scrambled_batch)
        _, pred_scrambled = torch.max(outputs_scrambled.data, 1)
        correct_scrambled += (pred_scrambled == labels).sum().item()

        total += labels.size(0)

print("\nEVALUATION TARGET RESULTS:")
print(f"Baseline Clean Test Accuracy: {100 * correct_clean / total:.2f}%")
print(f"Scrambled Jigsaw Test Accuracy: {100 * correct_scrambled / total:.2f}%")


EVALUATION TARGET RESULTS:
Baseline Clean Test Accuracy: 70.64%
Scrambled Jigsaw Test Accuracy: 31.83%
